In [51]:
import copy
from typing import Dict

import duckdb
import pandas as pd
import numpy as np
from numpy.typing import ArrayLike
from sklearn.preprocessing import OrdinalEncoder
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [52]:
with duckdb.connect("/home/maria-khanina/mkgit/movies_project/data/database.duckdb", read_only=True) as db_conn:
    ratings_df = db_conn.execute("""
        SELECT *
        FROM ratings
            INNER JOIN movies USING (movie_id)
            INNER JOIN users USING (user_id)
    """).df()

In [53]:
ratings_df.head()

,user_id,movie_id,rating,timestamp,title,genres,gender,age,occupation,zip_code
0,793,2407,2,975421808,Cocoon (1985),Comedy|Sci-Fi,F,35,14,37043
1,793,2268,4,975422633,"Few Good Men, A (1992)",Crime|Drama,F,35,14,37043
2,793,933,3,975423990,To Catch a Thief (1955),Comedy|Romance|Thriller,F,35,14,37043
3,793,3217,2,975423165,"Star Is Born, A (1937)",Drama,F,35,14,37043
4,793,1617,3,975420951,L.A. Confidential (1997),Crime|Film-Noir|Mystery|Thriller,F,35,14,37043


In [54]:
ratings_df["age"].value_counts()

age
25    395556
35    199003
18    183536
45     83633
50     72490
56     38780
1      27211
Name: count, dtype: int64

In [ ]:
class Ratings(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        is_training: bool = True,
    ) -> None:
        self.df = df.reset_index(drop=True)
        self.is_training = is_training

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        row = self.df.iloc[idx]
        
        user_id = int(row["user_id"])
        
        # if self.is_training and torch.rand(1,).item() < 0.05:
            # user_id = 0
            
        return {
            "user_id": torch.tensor(user_id, dtype=torch.long),
            "movie_id": torch.tensor(row["movie_id"], dtype=torch.long),
            "age": torch.tensor(row["age"], dtype=torch.long),
            "gender": torch.tensor(row["gender"], dtype=torch.long),
            "job": torch.tensor(row["occupation"], dtype=torch.long),
            "rating": torch.tensor(row["rating"], dtype=torch.float32),
        }


class Ranker(nn.Module):
    def __init__(
        self,
        users_num: int,
        movies_num: int,
        ages_num: int,
        genders_num: int,
        jobs_num: int,
        lin_layers_dims: list[int] = [128, 64, 32],
        dropout_ratio=0.2,
        user_dim=32,
        movie_dim=32,
        age_dim=4,
        gender_dim=2,
        job_dim=8,
    ) -> None:
        super().__init__()
        
        self.user_embedding = nn.Embedding(
            num_embeddings=users_num + 1,
            embedding_dim=user_dim,
        )
        self.movie_embedding = nn.Embedding(
            num_embeddings=movies_num + 1,
            embedding_dim=movie_dim,
        )
        self.job_embedding = nn.Embedding(
            num_embeddings=jobs_num + 1,
            embedding_dim=job_dim,
        )
        self.age_embedding = nn.Embedding(
            num_embeddings=ages_num + 1,
            embedding_dim=age_dim,
        )
        self.gender_embedding = nn.Embedding(
            num_embeddings=genders_num + 1,
            embedding_dim=gender_dim,
        )
        
        input_dim = user_dim + movie_dim + job_dim + age_dim + gender_dim
        
        layers = []
        curr_dim = input_dim
        
        for ind, dim in enumerate(lin_layers_dims):
            layers.append(nn.Linear(curr_dim, dim))
            layers.append(nn.SiLU())
            layers.append(nn.BatchNorm1d(dim))
            
            if ind == 0:
                layers.append(nn.Dropout(p=dropout_ratio))
                
            curr_dim = dim
            
        layers.append(nn.Linear(curr_dim, 1))
        
        self.net = nn.Sequential(*layers)

    def forward(
        self,
        user_ids: torch.Tensor,
        movie_ids: torch.Tensor,
        ages: torch.Tensor,
        genders: torch.Tensor,
        jobs: torch.Tensor,
    ) -> torch.Tensor:
        
        user_vector = self.user_embedding(user_ids)
        movie_vector = self.movie_embedding(movie_ids)
        age_vector = self.age_embedding(ages)
        gender_vector = self.gender_embedding(genders)
        job_vector = self.job_embedding(jobs)
        
        cat_features = torch.cat(
            [
                user_vector,
                movie_vector,
                age_vector,
                gender_vector,
                job_vector,
            ],
            dim=1,
        )
        
        return self.net(cat_features).squeeze()


def _run_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    is_train: bool,
    optimizer: optim.Optimizer = None,
) -> float:
    if is_train:
        model.train()
    else:
        model.eval()
    
    epoch_loss = 0.0
    dataset_len = 0
    
    with torch.set_grad_enabled(is_train,):
        for batch in dataloader:
            user_ids = batch["user_id"].to(device)
            movie_ids = batch["movie_id"].to(device)
            ages = batch["age"].to(device)
            genders = batch["gender"].to(device)
            jobs = batch["job"].to(device)
            y_true = batch["rating"].to(device).float()

            y_pred = model(
                user_ids,
                movie_ids,
                ages,
                genders,
                jobs,
            )
            
            loss = criterion(y_pred, y_true)
            
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
            epoch_loss += loss.item() * user_ids.size(0)
            dataset_len += user_ids.size(0)
            
    return epoch_loss / dataset_len


def train_ranker(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs=10_000,
    learning_rate=0.001,
    patience=10,
    tol=1e-4,
    weight_decay=1e-5,
) -> None:
    """if torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")"""
    device = torch.device("cpu")
    model = model.to(device)
    
    criterion = nn.MSELoss()
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=learning_rate,
        weight_decay=weight_decay,
    )
    
    best_val_loss = np.inf
    to_beat_loss = np.inf
    no_improve_count = 0
    best_weights = None
    
    for epoch in range(epochs):
        avg_train_loss = _run_epoch(
            model=model,
            dataloader=train_loader,
            criterion=criterion,
            device=device,
            is_train=True,
            optimizer=optimizer,
        )
        
        avg_val_loss = _run_epoch(
            model=model,
            dataloader=val_loader,
            criterion=criterion,
            device=device,
            is_train=False,
        )
        
        print(
            f"Epoch {epoch+1:05d}/{epochs}, "
            f"Train RMSE: {np.sqrt(avg_train_loss):.4f}, "
            f"Val RMSE: {np.sqrt(avg_val_loss):.4f}."
        )
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_weights = copy.deepcopy(model.state_dict())

        if avg_val_loss < (to_beat_loss - tol):
            to_beat_loss = avg_val_loss
            no_improve_count = 0
        else:
            no_improve_count += 1
            
        if no_improve_count >= patience:
            break
    
    if best_weights is not None:
        model.load_state_dict(best_weights)

In [56]:
def per_var_split_mask(
        df_sorted: pd.DataFrame,
        var: str,
        max_test_ratio=0.1,
        test_max_per_var=10,
):
    """
        Prepare splitting for given dataframe such as each unique value of var 
        has max_test_ratio ratio of records in test set capped at test_max_per_var.
    """
    per_var_count = df_sorted[var].map(df_sorted[var].value_counts())
    per_var_ranks = df_sorted.groupby(var).cumcount(ascending=False)
    per_var_test_size = np.minimum(test_max_per_var, max_test_ratio * per_var_count)
    per_var_test_size = np.maximum(per_var_test_size, 3)
    test_mask = per_var_ranks < per_var_test_size

    return test_mask

In [57]:
def per_var_temporal_split(
        X: pd.DataFrame,
        var="user_id",
        time_col="timestamp",
        max_test_ratio=0.1,
        test_max_per_var=10,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """
        Split X and y to train and test such as for each value of var
        all test records have greater time_col values than train records,
        and each value of var receives max_test_ratio of records in test,
        capped at test_max_per_var.
    """
    sort_indices = X.sort_values(by=[var, time_col]).index
    X_sorted = X.loc[sort_indices]
    test_mask = per_var_split_mask(X_sorted, var, max_test_ratio, test_max_per_var)
    return X_sorted[~test_mask], X_sorted[test_mask]

In [58]:
train_all_df, test_df = per_var_temporal_split(
    X=ratings_df,
    var="user_id",
    time_col="timestamp",
    max_test_ratio=0.1,
    test_max_per_var=10,
)

In [59]:
train_big_df, outer_val_df = per_var_temporal_split(
    train_all_df,
    var="user_id",
    time_col="timestamp",
    max_test_ratio=0.1,
    test_max_per_var=10,
)

In [60]:
train_df, val_df = per_var_temporal_split(
    train_big_df,
    var="user_id",
    time_col="timestamp",
    max_test_ratio=0.1,
    test_max_per_var=10,
)

In [61]:
train_all_df.groupby("user_id").size().describe()

count    6040.000000
mean      157.940728
std       191.219097
min        17.000000
25%        39.000000
50%        86.000000
75%       198.000000
max      2304.000000
dtype: float64

In [62]:
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

cat_cols = ["gender", "occupation"]

train_df[cat_cols] = encoder.fit_transform(train_df[cat_cols]) + 1

val_df[cat_cols] = encoder.transform(val_df[cat_cols]) + 1
outer_val_df[cat_cols] = encoder.transform(outer_val_df[cat_cols]) + 1
test_df[cat_cols] = encoder.transform(test_df[cat_cols]) + 1

In [63]:
train_df

,user_id,movie_id,rating,timestamp,title,genres,gender,age,occupation,zip_code
4127,1,3186,4,978300019,"Girl, Interrupted (1999)",Drama,1.0,1,11.0,48067
4118,1,1270,5,978300055,Back to the Future (1985),Comedy|Sci-Fi,1.0,1,11.0,48067
4123,1,1721,4,978300055,Titanic (1997),Drama|Romance,1.0,1,11.0,48067
4133,1,1022,5,978300055,Cinderella (1950),Animation|Children's|Musical,1.0,1,11.0,48067
4120,1,2340,3,978300103,Meet Joe Black (1998),Romance,1.0,1,11.0,48067
...,...,...,...,...,...,...,...,...,...,...
817665,6040,1912,3,964828542,Out of Sight (1998),Action|Crime|Romance,2.0,25,7.0,11106
817862,6040,3703,4,964828575,Mad Max 2 (a.k.a. The Road Warrior) (1981),Action|Sci-Fi,2.0,25,7.0,11106
817661,6040,1594,3,964828599,In the Company of Men (1997),Drama,2.0,25,7.0,11106
817701,6040,39,3,964828636,Clueless (1995),Comedy|Romance,2.0,25,7.0,11106


In [64]:
max_user = int(train_df["user_id"].max())
max_movie = int(train_df["movie_id"].max())
max_age = int(train_df["age"].max())
max_gender = int(train_df["gender"].max())
max_job = int(train_df["occupation"].max())

In [65]:
max_user, max_movie, max_age, max_gender, max_job

(6040, 3952, 56, 2, 21)

In [66]:
train_dataset = Ratings(train_df, is_training=True)
val_dataset = Ratings(val_df, is_training=False)

train_loader = DataLoader(
    train_dataset, 
    batch_size=1024, 
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=1024, 
    shuffle=False,
)

ranker_model = Ranker(
    users_num=max_user,
    movies_num=max_movie,
    ages_num=max_age,
    genders_num=max_gender,
    jobs_num=max_job,
)

train_ranker(
    model=ranker_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=10_000,
    learning_rate=0.01,
)

KeyboardInterrupt: 